In [ ]:

"""
main.py

End-to-end orchestration: data -> features -> walk-forward models ->
backtest comparison. Run this after data_prep.py has pulled CRSP prices
and point-in-time S&P 500 membership from WRDS.

    python src/data_prep.py <wrds_username>   # one-time WRDS pull, caches to data/
    python src/main.py                        # runs the full research pipeline
"""

# for ensuring that the notebook imports the updated function files instead of using the one in cache
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path
from tqdm import tqdm

from features import *
from model import run_walk_forward, summarize_ic
from backtest import compare_models, performance_summary, compute_portfolio_returns
from visualize import plot_model_comparison


In [3]:
# dynamic directory
DATA_DIR = Path.cwd().parent / "data"
OUTPUT_DIR = Path.cwd().parent / "output"


In [4]:

# Forward-return horizon in months. test_months/step_months are set equal to
# HORIZON below so each walk-forward test period's realized return window is
# disjoint from the next -- required for backtest.py to validly compound
# them as a sequential return series. See model.run_walk_forward's docstring.
HORIZON = 3

print("Loading CRSP price panel and point-in-time membership...")
daily_panel = pd.read_parquet(DATA_DIR / "prices_wrds.parquet")
membership = pd.read_parquet(DATA_DIR / "sp500_membership.parquet")


Loading CRSP price panel and point-in-time membership...


In [6]:
monthly = resample_to_monthly(daily_panel)
monthly

,permno,cum_ret_index,mkt_cap,avg_dollar_vol,daily_ret_std,date
0,10104,1.024960,1.618290e+11,9.281028e+08,0.010320,2011-01-31
183,10107,0.993373,2.329560e+11,1.934771e+09,0.014022,2011-01-31
366,10137,1.063530,4.381027e+09,4.474958e+07,0.008353,2011-01-31
368,10138,1.021381,1.705746e+10,1.130792e+08,0.014500,2011-01-31
551,10145,1.053612,4.369749e+10,2.309556e+08,0.009469,2011-01-31
...,...,...,...,...,...,...
91938,93096,2.742910,2.614743e+10,4.910172e+08,0.022238,2026-03-31
92028,93132,5.135102,6.046659e+10,5.000115e+08,0.019367,2026-03-31
92134,93246,0.639111,1.146118e+10,2.224056e+08,0.029018,2026-03-31
92297,93429,4.069827,2.941903e+10,2.675519e+08,0.017722,2026-03-31


In [ ]:
monthly = add_momentum_features(monthly)
monthly

,permno,cum_ret_index,mkt_cap,avg_dollar_vol,daily_ret_std,date,mom_1m,mom_3m,mom_12m_ex1
0,10104,1.024960,1.618290e+11,9.281028e+08,0.010320,2011-01-31,NaN,NaN,NaN
1,10104,1.052799,1.665069e+11,6.992905e+08,0.015679,2011-02-28,0.027162,NaN,NaN
2,10104,1.069838,1.691857e+11,9.353107e+08,0.018507,2011-03-31,0.016184,NaN,NaN
3,10104,1.152763,1.819762e+11,9.234382e+08,0.009951,2011-04-30,0.077511,0.124690,NaN
4,10104,1.096986,1.734270e+11,8.471734e+08,0.015775,2011-05-31,-0.048385,0.041970,NaN
...,...,...,...,...,...,...,...,...,...
92357,93436,1.856851,1.430668e+12,3.582897e+10,0.033233,2025-11-30,-0.057801,0.288436,0.322753
92358,93436,1.941239,1.686900e+12,3.409953e+10,0.023688,2025-12-31,0.045447,0.011243,0.065202
92359,93436,1.857889,1.615084e+12,2.748052e+10,0.025053,2026-01-31,-0.042937,-0.057274,0.111522
92360,93436,1.737453,1.510391e+12,2.418032e+10,0.020240,2026-02-28,-0.064824,-0.064301,0.469084


In [18]:

print("Building feature panel...")
feature_panel = build_feature_panel(daily_panel, membership, horizon=HORIZON)
print(f"Feature panel shape: {feature_panel.shape}")
print(f"Date range: {feature_panel['date'].min()} to {feature_panel['date'].max()}")
print(
    f"Unique PERMNOs represented (point-in-time members only): "
    f"{feature_panel['permno'].nunique()}"
)

OUTPUT_DIR.mkdir(exist_ok=True)
feature_panel.to_parquet(OUTPUT_DIR / "feature_panel.parquet", index=False)


Building feature panel...
Feature panel shape: (80685, 18)
Date range: 2012-01-31 00:00:00 to 2025-12-31 00:00:00
Unique PERMNOs represented (point-in-time members only): 751


In [17]:
feature_panel

,permno,cum_ret_index,mkt_cap,avg_dollar_vol,realized_vol,date,mom_1m,mom_3m,mom_12m_ex1,log_mkt_cap,log_dollar_vol,fwd_ret,mom_1m_z,mom_3m_z,mom_12m_ex1_z,realized_vol_z,log_mkt_cap_z,log_dollar_vol_z
12,10104,0.909792,1.417789e+11,9.977141e+08,0.009604,2012-01-31,0.102246,-0.137243,-0.194702,25.677534,20.720977,0.044340,0.581911,-1.795552,-0.805056,-0.816627,2.412160,2.272625
13,10104,0.943493,1.456606e+11,8.185006e+08,0.011857,2012-02-29,0.037043,-0.064759,-0.135835,25.704545,20.522985,-0.093324,-0.067986,-1.451836,-0.648039,-0.334194,2.408946,2.108568
14,10104,0.940430,1.450741e+11,1.099563e+09,0.012438,2012-03-31,-0.003247,0.139365,-0.118097,25.700510,20.818178,0.020626,-0.467092,0.096863,-0.719984,-0.188357,2.356913,2.465878
15,10104,0.950132,1.462681e+11,7.758613e+08,0.011946,2012-04-30,0.010317,0.044340,-0.184195,25.708707,20.469484,0.029320,0.305929,-0.129027,-1.005198,-0.599295,2.363086,2.093286
16,10104,0.855443,1.298354e+11,7.986041e+08,0.014791,2012-05-31,-0.099659,-0.093324,-0.133870,25.589533,20.498376,0.198148,-0.407942,-0.330610,-0.723287,-0.117092,2.295597,2.097098
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92051,93436,1.441167,1.076881e+12,2.522249e+10,0.022692,2025-08-31,0.083044,-0.036338,0.439781,27.705090,23.951002,0.288436,0.761595,-0.787535,1.340828,0.659644,2.908778,3.669326
92052,93436,1.919656,1.478249e+12,3.744436e+10,0.028232,2025-09-30,0.332015,0.399987,0.276119,28.021880,24.346122,0.011243,4.149459,2.490408,0.829468,1.803334,3.141362,3.389105
92053,93436,1.970763,1.518436e+12,3.879408e+10,0.032817,2025-10-31,0.026623,0.481038,0.779952,28.048702,24.381533,-0.057274,0.458562,2.911796,2.464783,1.551479,3.135075,3.497685
92054,93436,1.856851,1.430668e+12,3.582897e+10,0.033233,2025-11-30,-0.057801,0.288436,0.322753,27.989162,24.302023,-0.064301,-1.027582,1.747141,1.025374,1.768506,3.075283,3.553019


In [ ]:

predictions_by_model = {}

for model_type in ["linear", "gbm"]:
    print(f"\nRunning walk-forward for model: {model_type}")
    preds = run_walk_forward(
        feature_panel, model_type=model_type,
        horizon=HORIZON, test_months=1, step_months=HORIZON,
    )
    predictions_by_model[model_type] = preds

    print(f"-- {model_type} IC summary --")
    summarize_ic(preds)

    preds.to_parquet(OUTPUT_DIR / f"predictions_{model_type}.parquet", index=False)

print("\n--- Model comparison ---")
comparison = compare_models(predictions_by_model, freq=12 // HORIZON)
print(comparison)
comparison.to_csv(OUTPUT_DIR / "model_comparison.csv")
plot_model_comparison(OUTPUT_DIR / "model_comparison.csv", OUTPUT_DIR / "model_comparison.png")
'''